In [1]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
CROP_SIZE = 512
NUM_CLASSES = 40
IGNORE_INDEX = 0
BATCH_SIZE = 2
NUM_WORKERS = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Batch Size:", BATCH_SIZE)
print("Crop Size:", CROP_SIZE)
print("Number of Classes:", NUM_CLASSES)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.14.1+cu130
Device: cuda
Batch Size: 2
Crop Size: 512
Number of Classes: 40
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU Memory: 6.0 GB


In [2]:
CACHE_DIR = r"C:\ReefSeg\data\raw\hf_cache"
METADATA_DIR = r"C:\ReefSeg\data\raw\metadata"

dataset = load_dataset(
    "josauder/coralscapesV2",
    cache_dir=CACHE_DIR
)

with open(os.path.join(METADATA_DIR, "class_mapping.json"), "r") as f:
    mapping_data = json.load(f)

fine_to_legacy = mapping_data["fine_to_legacy"]

print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))
print("Class mappings:", len(fine_to_legacy))

Train: 1790
Validation: 170
Test: 473
Class mappings: 95


In [3]:
def convert_mask(mask):
    mask_array = np.array(mask)
    converted_mask = np.zeros_like(mask_array)

    for fine_id, info in fine_to_legacy.items():
        converted_mask[mask_array == int(fine_id)] = int(info["legacy_id"])

    return converted_mask


IMAGE_MEAN = torch.tensor(
    [0.3155488, 0.51109336, 0.50385326]
).view(3, 1, 1)

IMAGE_STD = torch.tensor(
    [0.20077191, 0.1957371, 0.2042009]
).view(3, 1, 1)


def preprocess_image(image):
    image_array = np.array(image, dtype=np.float32)

    image_tensor = torch.from_numpy(image_array)
    image_tensor = image_tensor.permute(2, 0, 1)
    image_tensor = image_tensor / 255.0
    image_tensor = (image_tensor - IMAGE_MEAN) / IMAGE_STD

    return image_tensor


def preprocess_mask(mask):
    mask_array = convert_mask(mask)
    mask_tensor = torch.from_numpy(mask_array.astype(np.int64))

    return mask_tensor

In [4]:
class ReefSegDataset(Dataset):
    def __init__(self, hf_split, crop_size=512):
        self.hf_split = hf_split
        self.crop_size = crop_size

        image_width, image_height = hf_split[0]["image"].size

        self.crops_per_row = image_width // crop_size
        self.crops_per_column = image_height // crop_size
        self.crops_per_image = self.crops_per_row * self.crops_per_column

    def __len__(self):
        return len(self.hf_split) * self.crops_per_image

    def __getitem__(self, idx):
        image_index = idx // self.crops_per_image
        crop_index = idx % self.crops_per_image

        row = crop_index // self.crops_per_row
        col = crop_index % self.crops_per_row

        left = col * self.crop_size
        top = row * self.crop_size

        sample = self.hf_split[image_index]

        image_crop = sample["image"].crop((
            left, top,
            left + self.crop_size,
            top + self.crop_size
        ))

        mask_crop = sample["label"].crop((
            left, top,
            left + self.crop_size,
            top + self.crop_size
        ))

        image_tensor = preprocess_image(image_crop)
        mask_tensor = preprocess_mask(mask_crop)

        return image_tensor, mask_tensor

In [5]:
train_dataset = ReefSegDataset(dataset["train"], CROP_SIZE)
val_dataset = ReefSegDataset(dataset["validation"], CROP_SIZE)
test_dataset = ReefSegDataset(dataset["test"], CROP_SIZE)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

image, mask = train_dataset[0]

print("\nImage shape:", image.shape)
print("Mask shape:", mask.shape)
print("Image dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

Train samples: 14320
Validation samples: 1360
Test samples: 3784

Image shape: torch.Size([3, 512, 512])
Mask shape: torch.Size([512, 512])
Image dtype: torch.float32
Mask dtype: torch.int64


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 7160
Validation batches: 680
Test batches: 1892


In [7]:
images, masks = next(iter(train_loader))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

print("Image range:", images.min().item(), images.max().item())
print("Mask classes:", torch.unique(masks))

Images shape: torch.Size([2, 3, 512, 512])
Masks shape: torch.Size([2, 512, 512])
Image dtype: torch.float32
Mask dtype: torch.int64
Image range: -1.7376694679260254 3.2333061695098877
Mask classes: tensor([ 0,  7, 12, 16, 21, 26, 31])


In [8]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2, 2)

        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        x4 = self.enc4(self.pool(x3))

        return x1, x2, x3, x4


class Bottleneck(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2, 2)
        self.conv = DoubleConv(256, 512)

    def forward(self, x):
        return self.conv(self.pool(x))


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = DoubleConv(64, 32)

    def forward(self, x, features):
        x1, x2, x3, x4 = features

        x = self.up4(x)
        x = torch.cat([x, x4], dim=1)
        x = self.dec4(x)

        x = self.up3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.dec3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.dec1(x)

        return x


class OutputLayer(nn.Module):
    def __init__(self, in_channels=32, num_classes=40):
        super().__init__()

        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, num_classes=40):
        super().__init__()

        self.encoder = Encoder()
        self.bottleneck = Bottleneck()
        self.decoder = Decoder()
        self.output_layer = OutputLayer(32, num_classes)

    def forward(self, x):
        features = self.encoder(x)
        bottleneck_output = self.bottleneck(features[-1])
        decoder_output = self.decoder(bottleneck_output, features)

        return self.output_layer(decoder_output)


model = UNet(num_classes=NUM_CLASSES).to(DEVICE)

model.eval()

with torch.no_grad():
    test_images = images[:1].to(DEVICE)
    test_output = model(test_images)

total_params = sum(p.numel() for p in model.parameters())

print("Output shape:", test_output.shape)
print("Parameters:", f"{total_params:,}")
print("Device:", next(model.parameters()).device)

Output shape: torch.Size([1, 40, 512, 512])
Parameters: 7,764,328
Device: cuda:0


In [9]:
criterion_ce = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

print("Cross Entropy Loss hazır.")
print("Ignore Index:", IGNORE_INDEX)

Cross Entropy Loss hazır.
Ignore Index: 0


In [10]:
def dice_loss(logits, targets, smooth=1.0):
    probs = torch.softmax(logits, dim=1)[:, 1:, :, :]

    valid_mask = (targets != IGNORE_INDEX).unsqueeze(1)

    safe_targets = targets.clamp(min=1) - 1

    target_one_hot = F.one_hot(
        safe_targets,
        num_classes=NUM_CLASSES - 1
    ).permute(0, 3, 1, 2).float()

    valid_mask = valid_mask.float()

    probs = probs * valid_mask
    target_one_hot = target_one_hot * valid_mask

    intersection = (probs * target_one_hot).sum(dim=(0, 2, 3))

    denominator = (
        probs.sum(dim=(0, 2, 3)) +
        target_one_hot.sum(dim=(0, 2, 3))
    )

    dice_scores = (2 * intersection + smooth) / (denominator + smooth)

    present_classes = target_one_hot.sum(dim=(0, 2, 3)) > 0

    if present_classes.any():
        loss = 1 - dice_scores[present_classes].mean()
    else:
        loss = logits.sum() * 0.0

    return loss

In [11]:
CE_WEIGHT = 0.5
DICE_WEIGHT = 0.5

def combined_loss(logits, targets):
    ce = criterion_ce(logits, targets)
    dice = dice_loss(logits, targets)

    total = CE_WEIGHT * ce + DICE_WEIGHT * dice

    return total, ce.detach(), dice.detach()

In [12]:
model.eval()

loss_images, loss_masks = next(iter(train_loader))

loss_images = loss_images[:1].to(DEVICE)
loss_masks = loss_masks[:1].to(DEVICE)

with torch.no_grad():
    logits = model(loss_images)

    total, ce, dice = combined_loss(logits, loss_masks)

print("Cross Entropy Loss:", ce.item())
print("Dice Loss:", dice.item())
print("Total Loss:", total.item())

Cross Entropy Loss: 3.575394630432129
Dice Loss: 0.9607502222061157
Total Loss: 2.2680723667144775


In [13]:
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.00001

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning Rate:", LEARNING_RATE)
print("Weight Decay:", WEIGHT_DECAY)

Optimizer: Adam
Learning Rate: 0.0001
Weight Decay: 1e-05


In [14]:
BATCH_SIZE = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Batch Size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Batch Size: 1
Train batches: 14320
Validation batches: 1360
Test batches: 3784


In [15]:
AMP_ENABLED = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)

print("AMP Enabled:", AMP_ENABLED)
print("GradScaler Enabled:", scaler.is_enabled())

if torch.cuda.is_available():
    print(
        "GPU Memory Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 3),
        "GB"
    )

AMP Enabled: True
GradScaler Enabled: True
GPU Memory Allocated: 0.115 GB


In [20]:
def cross_entropy_loss(logits, targets):
    semantic_logits = logits[:, 1:, :, :].float()

    valid_mask = targets != IGNORE_INDEX
    safe_targets = (targets - 1).clamp(min=0)

    pixel_losses = F.cross_entropy(
        semantic_logits,
        safe_targets,
        reduction="none"
    )

    loss = (pixel_losses * valid_mask).sum() / valid_mask.sum().clamp(min=1)

    return loss


def dice_loss(logits, targets, smooth=1.0):
    semantic_logits = logits[:, 1:, :, :].float()
    probs = torch.softmax(semantic_logits, dim=1)

    valid_mask = (targets != IGNORE_INDEX).unsqueeze(1)

    safe_targets = (targets - 1).clamp(min=0)

    target_one_hot = F.one_hot(
        safe_targets,
        num_classes=NUM_CLASSES - 1
    ).permute(0, 3, 1, 2).float()

    valid_mask = valid_mask.float()

    probs = probs * valid_mask
    target_one_hot = target_one_hot * valid_mask

    intersection = (probs * target_one_hot).sum(dim=(0, 2, 3))

    denominator = (
        probs.sum(dim=(0, 2, 3)) +
        target_one_hot.sum(dim=(0, 2, 3))
    )

    dice_scores = (2 * intersection + smooth) / (denominator + smooth)

    present_classes = target_one_hot.sum(dim=(0, 2, 3)) > 0

    if present_classes.any():
        loss = 1 - dice_scores[present_classes].mean()
    else:
        loss = logits.sum() * 0.0

    return loss


def combined_loss(logits, targets):
    ce = cross_entropy_loss(logits, targets)
    dice = dice_loss(logits, targets)

    total = CE_WEIGHT * ce + DICE_WEIGHT * dice

    return total, ce.detach(), dice.detach()

In [21]:
loss_images, loss_masks = next(iter(train_loader))

loss_images = loss_images.to(DEVICE)
loss_masks = loss_masks.to(DEVICE)

model.eval()

with torch.no_grad():
    logits = model(loss_images)
    total, ce, dice = combined_loss(logits, loss_masks)

print("Cross Entropy Loss:", ce.item())
print("Dice Loss:", dice.item())
print("Total Loss:", total.item())
print("Ignored pixels:", (loss_masks == IGNORE_INDEX).sum().item())

Cross Entropy Loss: 3.6924052238464355
Dice Loss: 0.9776800870895386
Total Loss: 2.335042715072632
Ignored pixels: 1448


In [22]:
model.train()

train_images, train_masks = next(iter(train_loader))

train_images = train_images.to(DEVICE)
train_masks = train_masks.to(DEVICE)

optimizer.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=AMP_ENABLED
):
    train_logits = model(train_images)

train_loss, train_ce, train_dice = combined_loss(
    train_logits,
    train_masks
)

scaler.scale(train_loss).backward()

scaler.step(optimizer)
scaler.update()

print("Cross Entropy:", train_ce.item())
print("Dice Loss:", train_dice.item())
print("Total Loss:", train_loss.item())

if torch.cuda.is_available():
    print(
        "Peak GPU Memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

del train_logits, train_loss, train_images, train_masks

Cross Entropy: 3.8980979919433594
Dice Loss: 0.9758744835853577
Total Loss: 2.436986207962036
Peak GPU Memory: 0.77 GB


In [23]:
def train_one_epoch(loader, max_batches=None):
    model.train()

    total_loss = 0
    total_ce = 0
    total_dice = 0
    batch_count = 0

    for batch_idx, (images, masks) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):
            logits = model(images)

        loss, ce, dice = combined_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_ce += ce.item()
        total_dice += dice.item()
        batch_count += 1

        if (batch_idx + 1) % 100 == 0:
            print(f"Batch: {batch_idx + 1}/{len(loader)} | Loss: {loss.item():.4f}")

    return {
        "loss": total_loss / batch_count,
        "ce": total_ce / batch_count,
        "dice": total_dice / batch_count
    }

In [24]:
@torch.no_grad()
def validate_one_epoch(loader, max_batches=None):
    model.eval()

    total_loss = 0
    total_ce = 0
    total_dice = 0
    batch_count = 0

    for batch_idx, (images, masks) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):
            logits = model(images)

        loss, ce, dice = combined_loss(logits, masks)

        total_loss += loss.item()
        total_ce += ce.item()
        total_dice += dice.item()
        batch_count += 1

    return {
        "loss": total_loss / batch_count,
        "ce": total_ce / batch_count,
        "dice": total_dice / batch_count
    }

In [25]:
train_results = train_one_epoch(train_loader, max_batches=5)
val_results = validate_one_epoch(val_loader, max_batches=3)

print("TRAIN RESULTS")
print(train_results)

print("\nVALIDATION RESULTS")
print(val_results)

TRAIN RESULTS
{'loss': 2.2430506706237794, 'ce': 3.5284164428710936, 'dice': 0.9576849341392517}

VALIDATION RESULTS
{'loss': 2.3335163593292236, 'ce': 3.7015767892201743, 'dice': 0.9654560287793478}


In [26]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = UNet(num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)

print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Optimizer:", optimizer.__class__.__name__)
print("Learning Rate:", LEARNING_RATE)
print("AMP:", AMP_ENABLED)
print("Device:", next(model.parameters()).device)

Model parameters: 7,764,328
Optimizer: Adam
Learning Rate: 0.0001
AMP: True
Device: cuda:0


In [ ]:
train_results = train_one_epoch(train_loader)
val_results = validate_one_epoch(val_loader)

print("\nTRAIN RESULTS")
print(train_results)

print("\nVALIDATION RESULTS")
print(val_results)

print(
    "\nPeak GPU Memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

Batch: 100/14320 | Loss: 2.6035
Batch: 200/14320 | Loss: 2.0386
Batch: 300/14320 | Loss: 1.9840
Batch: 400/14320 | Loss: 2.0351
Batch: 500/14320 | Loss: 1.8026
Batch: 600/14320 | Loss: 2.0135
Batch: 700/14320 | Loss: 2.0889
Batch: 800/14320 | Loss: 1.6820
Batch: 900/14320 | Loss: 1.8991
Batch: 1000/14320 | Loss: 1.6400
Batch: 1100/14320 | Loss: 1.9125
Batch: 1200/14320 | Loss: 1.8275
Batch: 1300/14320 | Loss: 1.9135
Batch: 1400/14320 | Loss: 1.6046
Batch: 1500/14320 | Loss: 1.3000
Batch: 1600/14320 | Loss: 1.6233
Batch: 1700/14320 | Loss: 1.4180
Batch: 1800/14320 | Loss: 2.2463
Batch: 1900/14320 | Loss: 1.3425
Batch: 2000/14320 | Loss: 1.6621
Batch: 2100/14320 | Loss: 1.3328
Batch: 2200/14320 | Loss: 2.3608
Batch: 2300/14320 | Loss: 1.8860
Batch: 2400/14320 | Loss: 1.3281
Batch: 2500/14320 | Loss: 1.1399
Batch: 2600/14320 | Loss: 1.3167
Batch: 2700/14320 | Loss: 1.6644
Batch: 2800/14320 | Loss: 1.7178
Batch: 2900/14320 | Loss: 1.2262
Batch: 3000/14320 | Loss: 1.0718
Batch: 3100/14320 |